# Occ4cast Lyft Training Walkthrough (Google Colab + Tesla T4)
This notebook is a step-by-step companion for running an Occ4cast baseline training job on the Lyft portion of OCFBench. It assumes you are using Google Colab with a single Tesla T4 GPU and that you are new to deep-learning projects. Follow the cells from top to bottom and read the explanatory notes along the way.

## 0. Prerequisites
Before opening the Colab runtime, make sure you have the following ready:
- **Google Drive space (\>\~300 GB)** to hold the raw Lyft Level 5 Perception dataset and the processed voxel files. The raw data alone is \>200 GB.
- **A Lyft Level 5 Perception dataset account** (apply at [https://level5.lyft.com/dataset/](https://level5.lyft.com/dataset/)) so you can download the raw data.
- **(Optional but recommended) A Hugging Face account + access token** if you plan to pull preprocessed samples from [ai4ce/OCFBench](https://huggingface.co/datasets/ai4ce/OCFBench). Large downloads work better over `git lfs` than through the browser.

## 1. Configure the Colab runtime
1. In Colab, navigate to **Runtime → Change runtime type**.
2. Choose **GPU** as the hardware accelerator (a Tesla T4 is the default free GPU).
3. Click **Save** and rerun this cell if you restart the runtime.

In [ ]:
#@title Verify that the GPU is visible
!nvidia-smi

## 2. Mount Google Drive (recommended)
Processing the Lyft dataset produces hundreds of gigabytes of voxel files. Mounting Drive lets you persist the processed data and training checkpoints between sessions. If you prefer local Colab storage (ephemeral), you can skip this and adjust the paths in later cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Update the two paths below to point to folders in your Drive. The **raw Lyft data** should contain the unzipped `train` split that you download from Lyft (with folders such as `data`, `maps`, etc.). The **processed root** is where Occ4cast will write KITTI-format intermediate files and final voxels.

In [ ]:
RAW_LYFT_ROOT = '/content/drive/MyDrive/datasets/lyft/lyft_level5/train'  # TODO: change me
PROCESSED_LYFT_ROOT = '/content/drive/MyDrive/datasets/lyft/lyft_kitti/train'  # TODO: change me
!mkdir -p "{PROCESSED_LYFT_ROOT}"

## 3. Clone Occ4cast and install dependencies
Occ4cast targets **Python 3.9** and **PyTorch 2.0.1 (CUDA 11.8)**. Colab currently ships with Python 3.10 and CUDA 12 toolkits, but PyTorch wheels compiled for CUDA 11.8 still run on a Tesla T4. We explicitly install the compatible versions below.

> **Tip:** If the wheel versions change in the future, visit [https://pytorch.org/get-started/locally/](https://pytorch.org/get-started/locally/) to find the matching command for your environment.

In [ ]:
!git clone https://github.com/ai4ce/Occ4cast.git
%cd Occ4cast

# Install PyTorch 2.0.1 + CUDA 11.8 wheels
!pip install --upgrade pip
!pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118

# Install project requirements
!pip install -r requirements.txt

# Install extra dependencies for Lyft preprocessing (replace cuXXX with cu118)
!sed -i 's/cuXXX/cu118/g' data_processing/lyft/requirements.txt
!pip install -r data_processing/lyft/requirements.txt

If you see compilation errors for `spconv`, rerun the cell above. Occasionally the download can time out; a retry usually succeeds. Confirm that `torch.cuda.is_available()` returns `True` before moving on.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU')
print('Torch version:', torch.__version__)

## 4. Prepare the Lyft dataset
You need the **train split** of the Lyft Level 5 perception dataset (the validation split is optional). Each split comes as multiple `.tar` archives. Download them to your computer or Google Drive, unzip them, and make sure the directory tree looks like `.../train/data`, `.../train/maps`, `.../train/lyft.json`, etc.

You now have two options:
1. **Local preprocessing:** Convert the raw Lyft data into Occ4cast voxels inside Colab (slower but works offline).
2. **Download preprocessed voxels from Hugging Face:** Faster but requires `git lfs` and enough quota.

### Option 1 — Convert raw Lyft data inside Colab
This step can take many hours. Feel free to break it into chunks by adjusting the `--start` and `--end` scene indices. The full training set contains 180 scenes; the default split uses 0-119 for training, 120-149 for validation, and 150-179 for testing.

In [ ]:
#@title Convert raw Lyft frames into voxels (this is very time-consuming)
START_SCENE = 0   #@param {type:"number"}
END_SCENE = 120   #@param {type:"number"}
AGGREGATE_PRE = 70   #@param {type:"number"}
AGGREGATE_POST = 70  #@param {type:"number"}
PREDICT_PRE = 10     #@param {type:"number"}
PREDICT_POST = 10    #@param {type:"number"}
USE_MULTIPROCESS = True  #@param {type:"boolean"}

convert_cmd = [
    'python', 'data_processing/lyft/convert_lyft.py',
    '-d', RAW_LYFT_ROOT,
    '-o', PROCESSED_LYFT_ROOT,
    '-s', str(int(START_SCENE)),
    '-e', str(int(END_SCENE)),
    '--a_pre', str(int(AGGREGATE_PRE)),
    '--a_post', str(int(AGGREGATE_POST)),
    '--p_pre', str(int(PREDICT_PRE)),
    '--p_post', str(int(PREDICT_POST)),
]
if USE_MULTIPROCESS:
    convert_cmd.append('--multi')

print('Running:', ' '.join(convert_cmd))
!{' '.join(convert_cmd)}

### Option 2 — Download preprocessed voxels
If you have access to the processed Lyft subset on Hugging Face (`ai4ce/OCFBench`), you can pull only the `voxel` folders to save time. The snippet below grabs the training scenes 0000-0179 and places them into `PROCESSED_LYFT_ROOT`. Adjust the paths or use `git lfs fetch` if you need partial downloads.

In [ ]:
#@title (Optional) Download preprocessed voxels from Hugging Face
USE_HF = False  #@param {type:"boolean"}
if USE_HF:
    !pip install -q huggingface_hub git-lfs
    !huggingface-cli login --token YOUR_TOKEN_HERE
    %cd $PROCESSED_LYFT_ROOT/..
    !git lfs install
    !git clone https://huggingface.co/datasets/ai4ce/OCFBench --branch main --depth 1 --single-branch ocfbench_tmp
    !rsync -av ocfbench_tmp/OCFBench-Lyft/voxel/ ./train/voxel/
    !rm -rf ocfbench_tmp
    %cd /content/Occ4cast
else:
    print('Skipping Hugging Face download. Set USE_HF=True to enable.')

After either option completes, verify that the processed directory has subfolders like `voxel/0000/0010.npz`. You should also see `calib`, `point_cloud`, and `image` directories if you ran the full converter.

In [ ]:
import os
from itertools import islice

processed_subdirs = sorted(os.listdir(PROCESSED_LYFT_ROOT))
print('Top-level processed folders:', processed_subdirs[:5])
voxel_dir = os.path.join(PROCESSED_LYFT_ROOT, 'voxel', '0000')
if os.path.isdir(voxel_dir):
    print('Sample voxel files:', list(islice(sorted(os.listdir(voxel_dir)), 5)))
else:
    print('Could not find voxel directory at', voxel_dir)

## 5. Launch a baseline training run
We can now train the 3D convolutional forecasting baseline on the processed Lyft voxels. A Tesla T4 has limited memory, so we start with:
- `--batch-size 1`
- `--p_pre 5 --p_post 5` (6 frames in, 6 frames predicted)
- Mixed precision (`--amp`) to reduce memory usage.

Feel free to increase these numbers gradually if you have headroom (watch the GPU usage in the Colab status bar).

In [ ]:
#@title Train Conv3D baseline on Lyft (adjust epochs for a full run)
BATCH_SIZE = 1    #@param {type:"number"}
NUM_EPOCHS = 5    #@param {type:"number"}
P_PRE = 5         #@param {type:"number"}
P_POST = 5        #@param {type:"number"}
NUM_WORKERS = 2   #@param {type:"number"}
USE_AMP = True    #@param {type:"boolean"}

train_cmd = [
    'python', 'baselines/train.py',
    '-d', 'lyft',
    '-r', PROCESSED_LYFT_ROOT,
    '-m', 'conv3d',
    '--batch-size', str(int(BATCH_SIZE)),
    '--num-epoch', str(int(NUM_EPOCHS)),
    '--num-workers', str(int(NUM_WORKERS)),
    '--p_pre', str(int(P_PRE)),
    '--p_post', str(int(P_POST)),
]
if USE_AMP:
    train_cmd.append('--amp')

print('Running:', ' '.join(train_cmd))
!{' '.join(train_cmd)}

Training logs and checkpoints are saved under `baselines/results/conv3d_lyft_p{P_PRE}{P_POST}_lr0.0005_batch{BATCH_SIZE}_amp/`. Each epoch logs train loss, validation precision/recall/IoU, and stores the best model by IoU along with `last.pth`.

In [ ]:
import glob, json
result_dirs = sorted(glob.glob('baselines/results/conv3d_lyft*'))
print('Found result directories:', result_dirs)
if result_dirs:
    latest = result_dirs[-1]
    print('Latest config:')
    with open(f'{latest}/config.json') as f:
        print(json.dumps(json.load(f), indent=2))
    print('Checkpoint files:', glob.glob(f'{latest}/ckpts/*.pth'))

## 6. Next steps
- Increase `NUM_EPOCHS`, `P_PRE`, `P_POST`, and `BATCH_SIZE` as your GPU memory allows.
        - Explore alternative models (`conv3d_softiou`, `convlstm`, `occ`) via the `-m` flag.
        - Use `baselines/run_eval.sh` with your saved checkpoints to evaluate on the validation or test split once training finishes.

Happy training!